# VisionTrack Analysis

This notebook documents the training, optimization, and validation workflow for VisionTrack.

## Scope
- Transfer learning with YOLO for person detection
- ONNX export and ONNX Runtime inference path
- Optional pruning workflow
- Evaluation metrics and audit checklist


In [ ]:
from pathlib import Path
import json

project_root = Path('.').resolve()
print('Project root:', project_root)
print('Config exists:', (project_root / 'models/checkpoints/config.yaml').exists())
print('Metrics exists:', (project_root / 'reports/performance_metrics.json').exists())

## 0) Exploratory Data Analysis (EDA)
Analyzing dataset distribution and validating preprocessing (resizing/normalization) before inference.

In [ ]:
import os
import glob
import cv2
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

dataset_dir = "data/coco_dataset"
images_dir = os.path.join(dataset_dir, "images", "train")
labels_dir = os.path.join(dataset_dir, "labels", "train")

class_counts = Counter()
label_files = glob.glob(os.path.join(labels_dir, "*.txt"))
if label_files:
    for lf in label_files:
        with open(lf, "r") as f:
            for line in f.readlines():
                cls_id = int(line.split()[0])
                class_counts[cls_id] += 1
    print("Class Distribution:", dict(class_counts))
else:
    print("No label files found. Please ensure the dataset is generated.")

In [ ]:
image_files = glob.glob(os.path.join(images_dir, "*.jpg"))
if image_files:
    image = cv2.imread(image_files[0])
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    target_size = (640, 640)
    resized_image = cv2.resize(image_rgb, target_size)
    normalized_image = resized_image.astype(np.float32) / 255.0
    
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(image_rgb)
    axs[0].set_title(f"Original {image_rgb.shape}")
    axs[0].axis("off")
    
    axs[1].imshow(normalized_image)
    axs[1].set_title(f"Preprocessed {normalized_image.shape}")
    axs[1].axis("off")
    plt.show()
else:
    print("No image files found.")

## 1) Transfer Learning Setup

Use a pre-trained YOLO model and fine-tune on your dataset in YOLO format.

Recommended command:

```bash
python3 models/train_yolo.py \
  --data data/coco_dataset/data.yaml \
  --model yolov8n.pt \
  --epochs 20 \
  --lr0 0.001
```

Notes:
- Minimum epochs target: 10+
- Small learning rate to preserve pre-trained features
- Best weights are copied to `models/checkpoints/best.pt`


In [ ]:
# Read and display training config/log artifacts
config_path = Path('models/checkpoints/config.yaml')
log_path = Path('models/checkpoints/training_log.json')

print('config.yaml exists:', config_path.exists())
print('training_log.json exists:', log_path.exists())
if log_path.exists():
    data = json.loads(log_path.read_text(encoding='utf-8'))
    print('training_log:', data)

## 2) ONNX Export + Runtime Path

Export command:

```bash
python3 models/export_onnx.py \
  --weights models/checkpoints/best.pt \
  --output models/checkpoints/best_quantized.onnx
```

The Streamlit app supports backend switching:
- `PyTorch` backend
- `ONNX Runtime` backend

This enables performance testing and comparison in deployment-like conditions.


In [ ]:
onnx_path = Path('models/checkpoints/best_quantized.onnx')
print('ONNX artifact exists:', onnx_path.exists())

if onnx_path.exists():
    print('ONNX file size (MB):', round(onnx_path.stat().st_size / (1024 * 1024), 3))

## 3) Optional Pruning

Pruning command:

```bash
python3 models/prune_yolo.py \
  --weights models/checkpoints/best.pt \
  --output models/checkpoints/best_pruned.pt \
  --amount 0.2
```

Target outcome:
- Approx. 20-30% reduction in model size (depends on model + pruning strategy)
- Re-evaluate precision/recall/F1 after pruning to confirm acceptable quality


## 4) Evaluation Metrics

Compute precision/recall/F1 using YOLO-format validation labels:

```bash
python3 models/evaluate_yolo.py \
  --weights models/checkpoints/best.pt \
  --images-dir data/coco_dataset/images/val \
  --labels-dir data/coco_dataset/labels/val
```

Generated file:
- `reports/performance_metrics.json`


In [ ]:
metrics_path = Path('reports/performance_metrics.json')
thresholds = {
    'detection_precision': 0.85,
    'detection_recall': 0.80,
    'f1_score': 0.85,
    'average_fps_per_stream': 15.0,
}

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Metrics:', metrics)
    for key, target in thresholds.items():
        value = float(metrics.get(key, 0.0))
        print(f'{key}: {value:.4f} (target >= {target}) ->', 'PASS' if value >= target else 'FAIL')
else:
    print('Metrics file not found:', metrics_path)

## 5) Demo Artifacts and Final Validation

Generate the required demo outputs:

```bash
python3 generate_demos.py \
  --video path/to/your/demo_video.mp4 \
  --backend onnx
```

Run final validation report:

```bash
python3 validate_project.py
```

Expected demo outputs:
- `reports/demo_results/roi_counting_example.png`
- `reports/demo_results/multi_stream_demo.mp4`
